# RandomForest 기본 학습과 튜닝

> 이전 실험 기록: 아래 코드와 출력은 단일 7:3 분할에서 수행한 초기 RF 실험이다. 현재는 30회 외부 그룹 분할을 사용하는 `deal_model_paper_rf_baseline.ipynb`와 `deal_model_paper_rf_tuning.ipynb`로 대체했다. [현재 실행 순서](README.md)를 확인한다.

전처리 → RandomForest 기본 성능 확인 → GridSearchCV → 기본/튜닝 결과 비교 → 후보 저장 순서로 진행한다.

- 기존 전처리 노트북의 13개 컬럼, 행당 4개 Unknown, 서로 다른 마스킹 10세트를 그대로 사용한다.
- 동일한 원본 입력과 그 마스킹 변형은 같은 그룹에 묶어 Train/Test와 CV 사이에 섞이지 않게 한다.
- 튜닝 기준은 Brier Score, 분류 임계값은 0.5로 유지한다. AUC·Accuracy·Precision·Recall·F1·FP/FN도 함께 확인한다.
- RandomForest 한 종류만 사용한다. TabICL, 여러 모델을 결합하는 Stacking, 임계값 탐색은 실행하지 않는다.
- [원 논문](https://doi.org/10.1016/j.eswa.2016.11.010)의 최종 모델 종류인 RandomForest를 출발점으로 삼는다. 논문의 22개 컬럼·평가 분할·구현 설정을 그대로 재현하거나 공개된 학습 모델을 이어 학습하는 것은 아니다.
- scikit-learn RandomForest는 CPU에서 실행한다. 이 구현은 MPS를 지원하지 않는다.

## 1. 라이브러리

In [1]:
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    ParameterGrid,
    StratifiedGroupKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 1
CLASSIFICATION_THRESHOLD = 0.5
SCORING = "neg_brier_score"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

## 2. 기존 전처리 결과 불러오기

`deal_data_preprocessing.ipynb`만 실행한다. 이전 모델 비교·튜닝·앙상블 노트북은 실행하지 않는다.
원본 CSV 위치가 다르면 커널을 시작하기 전에 `SALESLUV_B2B_DATA_PATH` 환경변수로 지정한다.

In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

ipython = get_ipython()
assert ipython is not None, "Jupyter 커널에서 실행해야 합니다."
# 원본 데이터 행과 전처리 전체 출력을 이 노트북에 다시 저장하지 않는다.
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

X_train_raw = ipython.user_ns["X_train_raw"]
y_train = ipython.user_ns["y_train"]
train_group_ids = ipython.user_ns["train_group_ids"]
X_test_raw_sets = ipython.user_ns["X_test_raw_sets"]
y_test = ipython.user_ns["y_test"]
input_group_ids = ipython.user_ns["input_group_ids"]
MODEL_FEATURE_NAMES = ipython.user_ns["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = ipython.user_ns["CATEGORY_VALUES"]
SOURCE_SHA256 = ipython.user_ns["SOURCE_SHA256"]

# 입력 순서, 정답 정렬, 마스킹 개수와 Train/Test 그룹 분리를 검증한다.
assert list(X_train_raw.columns) == list(MODEL_FEATURE_NAMES)
assert X_train_raw.index.equals(y_train.index)
assert len(train_group_ids) == len(y_train)
assert X_train_raw.eq("Unknown").sum(axis=1).eq(4).all()
assert set(y_train.unique()) == {0, 1}
assert len(X_test_raw_sets) == 10
for X_test_raw in X_test_raw_sets.values():
    assert list(X_test_raw.columns) == list(MODEL_FEATURE_NAMES)
    assert X_test_raw.index.equals(y_test.index)
    assert X_test_raw.eq("Unknown").sum(axis=1).eq(4).all()
    assert set(train_group_ids).isdisjoint(input_group_ids.loc[X_test_raw.index])

# 기본 모델과 튜닝 후보가 모두 같은 검증 행을 평가하도록 폴드를 한 번만 만든다.
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv5.split(X_train_raw, y_train, groups=train_group_ids))
for train_index, valid_index in cv_splits:
    assert set(train_group_ids[train_index]).isdisjoint(set(train_group_ids[valid_index]))
    assert set(y_train.iloc[train_index].unique()) == {0, 1}
    assert set(y_train.iloc[valid_index].unique()) == {0, 1}

train_original_rows = X_train_raw.index.get_level_values("original_row_id").nunique()
print(f"Train: 원본 {train_original_rows}건 → 마스킹 포함 {len(y_train)}행")
print(f"Test: 같은 {len(y_test)}건에 서로 다른 마스킹 {len(X_test_raw_sets)}세트")
print(f"입력 {len(MODEL_FEATURE_NAMES)}개, CV {len(cv_splits)}-Fold, 임계값 0.5")

Train: 원본 313건 → 마스킹 포함 3130행
Test: 같은 135건에 서로 다른 마스킹 10세트
입력 13개, CV 5-Fold, 임계값 0.5


### 해석

Train의 10개 마스킹 변형은 새로운 거래 10건이 아니다. 같은 거래·같은 원본 입력이 검증 폴드로 넘어가면 성능이 부풀 수 있어 그룹을 유지한다.
Test 10세트도 같은 거래의 입력 조건을 바꾼 것이며, 세트 사이 점수 차이는 결측 위치에 대한 민감도를 의미한다.

## 3. RandomForest 기본 모델

In [3]:
# 고정된 범주 목록을 사용하므로 새 범주는 오류로 처리하고 Unknown은 정상 범주로 학습한다.
onehot = OneHotEncoder(
    categories=[list(CATEGORY_VALUES[column]) for column in MODEL_FEATURE_NAMES],
    drop="first",
    handle_unknown="error",
    sparse_output=False,
    dtype=np.float32,
)
model_rf = Pipeline(
    [
        ("onehot", onehot),
        ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
    ]
)

started = perf_counter()
cv_score = cross_val_score(
    model_rf,
    X_train_raw,
    y_train,
    cv=cv_splits,
    scoring=SCORING,
    n_jobs=-1,
    error_score="raise",
)
baseline_cv_seconds = perf_counter() - started

started = perf_counter()
model_rf.fit(X_train_raw, y_train)
baseline_fit_seconds = perf_counter() - started
print(f"기본 CV Brier: {-cv_score.mean():.6f}")
print(f"기본 CV 시간: {baseline_cv_seconds:.2f}초")
print(f"Train 전체 1회 학습: {baseline_fit_seconds:.2f}초")

기본 CV Brier: 0.220836
기본 CV 시간: 1.16초
Train 전체 1회 학습: 0.14초


### 해석

기본값은 나무 100개, 깊이 제한 없음, 잎의 최소 샘플 수 1, 분기 후보 피처 수 `sqrt`다.
Brier는 작을수록 좋다. scikit-learn의 점수 비교 규칙 때문에 계산에는 음수 값을 사용하고, 출력할 때 다시 양수로 표시한다.

## 4. GridSearchCV 튜닝

In [4]:
# 2 × 3 × 2 × 2 = 24개 조합만 탐색한다. 기본 모델의 설정도 포함한다.
param = {
    "classifier__n_estimators": [100, 300],
    "classifier__max_depth": [None, 6, 12],
    "classifier__min_samples_leaf": [1, 5],
    "classifier__max_features": ["sqrt", 0.5],
}
print(f"탐색: {len(ParameterGrid(param))}개 조합 × {len(cv_splits)}-Fold")

# GridSearch에서 병렬 처리하고 RF 내부는 1개 스레드로 두어 중첩 병렬화를 피한다.
search_rf = GridSearchCV(
    model_rf,
    param,
    cv=cv_splits,
    scoring=SCORING,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    error_score="raise",
)
started = perf_counter()
search_rf.fit(X_train_raw, y_train)
tuning_seconds = perf_counter() - started

# 기본 설정을 포함하고 같은 폴드를 사용하므로 선택된 CV 점수는 기본보다 나쁘지 않아야 한다.
assert search_rf.best_score_ >= cv_score.mean() - 1e-10
model_rf_tuned = search_rf.best_estimator_
print(f"최적 파라미터: {search_rf.best_params_}")
print(f"최적 CV Brier: {-search_rf.best_score_:.6f}")
print(f"튜닝 시간(최적 모델 재학습 포함): {tuning_seconds:.2f}초")

탐색: 24개 조합 × 5-Fold


최적 파라미터: {'classifier__max_depth': 12, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 300}
최적 CV Brier: 0.198926
튜닝 시간(최적 모델 재학습 포함): 4.31초


### 해석

`n_estimators`는 나무 수, `max_depth`는 나무 깊이, `min_samples_leaf`는 잎에 남겨야 하는 최소 샘플 수,
`max_features`는 분기 때 검토하는 피처 수를 조절한다.
Test를 보지 않고 Train 내부 CV Brier로 최적 파라미터를 선택한다. 같은 CV에서 개선되더라도 Test 개선까지 보장되지는 않는다.

In [5]:
cv_results = pd.DataFrame(
    {
        "rank": search_rf.cv_results_["rank_test_score"],
        "params": search_rf.cv_results_["params"],
        "cv_brier": -search_rf.cv_results_["mean_test_score"],
        "cv_brier_std": search_rf.cv_results_["std_test_score"],
        "train_brier": -search_rf.cv_results_["mean_train_score"],
        "mean_fit_seconds": search_rf.cv_results_["mean_fit_time"],
    }
).sort_values("rank", ignore_index=True)
display(cv_results.round(6))

,rank,params,cv_brier,cv_brier_std,train_brier,mean_fit_seconds
0,1,"{'classifier__max_depth': 12, 'classifier__max...",0.198926,0.020301,0.141388,0.303755
1,2,"{'classifier__max_depth': 6, 'classifier__max_...",0.198933,0.019202,0.170830,0.244676
2,3,"{'classifier__max_depth': 12, 'classifier__max...",0.199314,0.020373,0.141726,0.105928
3,4,"{'classifier__max_depth': None, 'classifier__m...",0.199394,0.020071,0.136075,0.339656
4,5,"{'classifier__max_depth': 6, 'classifier__max_...",0.199539,0.019738,0.170736,0.095015
5,6,"{'classifier__max_depth': 6, 'classifier__max_...",0.200000,0.019567,0.168093,0.257561
6,7,"{'classifier__max_depth': None, 'classifier__m...",0.200052,0.019449,0.136190,0.125722
7,8,"{'classifier__max_depth': 6, 'classifier__max_...",0.200461,0.019517,0.167860,0.093539
8,9,"{'classifier__max_depth': 6, 'classifier__max_...",0.203186,0.022547,0.158372,0.384716
9,10,"{'classifier__max_depth': 6, 'classifier__max_...",0.203321,0.022686,0.158628,0.132591


### 해석

`train_brier`는 학습한 행의 점수, `cv_brier`는 해당 폴드에서 학습에 쓰지 않은 그룹의 점수다.
학습 점수만 좋고 CV가 나쁘면 과적합을 의심한다. `mean_fit_seconds`는 병렬 탐색 중의 평균 학습 시간이며 단독 실행 속도와 같지는 않다.

## 5. 기본 모델과 튜닝 모델의 Test 비교

In [6]:
def evaluate_test_sets(estimator):
    """학습에 사용하지 않은 같은 Test 10세트를 평가한다."""
    rows = []
    won_index = list(estimator.classes_).index(1)
    for set_name, X_test_raw in X_test_raw_sets.items():
        probability = estimator.predict_proba(X_test_raw)[:, won_index]
        prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
        rows.append(
            {
                "mask_set": set_name,
                "brier": brier_score_loss(y_test, probability),
                "auc": roc_auc_score(y_test, probability),
                "accuracy": accuracy_score(y_test, prediction),
                "precision": precision_score(y_test, prediction, zero_division=0),
                "recall": recall_score(y_test, prediction, zero_division=0),
                "f1": f1_score(y_test, prediction, zero_division=0),
                "fp": int(fp),
                "fn": int(fn),
                "tn": int(tn),
                "tp": int(tp),
            }
        )
    return pd.DataFrame(rows).set_index("mask_set")


test_baseline = evaluate_test_sets(model_rf)
test_tuned = evaluate_test_sets(model_rf_tuned)
test_comparison = pd.DataFrame(
    [test_baseline.mean(), test_tuned.mean()],
    index=["RandomForest_base", "RandomForest_tuned"],
)
test_comparison.insert(0, "cv_brier", [-cv_score.mean(), -search_rf.best_score_])
display(test_comparison.round(6))
display(test_tuned.round(6))

,cv_brier,brier,auc,accuracy,precision,recall,f1,fp,fn,tn,tp
RandomForest_base,0.220836,0.205182,0.747267,0.690370,0.691888,0.695588,0.692792,21.1,20.7,45.9,47.3
RandomForest_tuned,0.198926,0.182146,0.804500,0.735556,0.705184,0.817647,0.756943,23.3,12.4,43.7,55.6


,brier,auc,accuracy,precision,recall,f1,fp,fn,tn,tp
mask_set,,,,,,,,,,
test_mask_01,0.190661,0.775461,0.725926,0.706667,0.779412,0.741259,22,15,45,53
test_mask_02,0.176434,0.825285,0.725926,0.686747,0.838235,0.754967,26,11,41,57
test_mask_03,0.186314,0.791703,0.725926,0.696203,0.808824,0.748299,24,13,43,55
test_mask_04,0.181032,0.805312,0.762963,0.725000,0.852941,0.783784,22,10,45,58
test_mask_05,0.191160,0.784789,0.711111,0.679012,0.808824,0.738255,26,13,41,55
test_mask_06,0.188936,0.784460,0.703704,0.679487,0.779412,0.726027,25,15,42,53
test_mask_07,0.170707,0.831651,0.762963,0.725000,0.852941,0.783784,22,10,45,58
test_mask_08,0.183229,0.795215,0.733333,0.690476,0.852941,0.763158,26,10,41,58
test_mask_09,0.178464,0.818810,0.740741,0.720000,0.794118,0.755245,21,14,46,54


### 해석

첫 표는 Test 10세트의 평균, 두 번째 표는 튜닝 모델의 마스킹 세트별 결과다.
FP는 실제 Lost를 Won으로 예측한 건수이며 위험 거래를 놓칠 수 있다는 점에서 주의한다.
FP만 줄이기 위해 임계값을 바꾸지는 않으며 FN·Recall·Accuracy도 함께 읽는다.
이 표로 파라미터를 다시 고르지 않고, 저장할 후보는 위 CV에서 정한 모델로 고정한다.

## 6. 단건 예측 시간 확인과 후보 저장

In [7]:
# 원본 영업 행을 저장하지 않고 허용 범주로 만든 합성 입력만 재로드 검증에 사용한다.
known_row = {
    column: next(value for value in CATEGORY_VALUES[column] if value != "Unknown")
    for column in MODEL_FEATURE_NAMES
}
masked_row = {**known_row, **dict.fromkeys(MODEL_FEATURE_NAMES[:4], "Unknown")}
self_check_X = pd.DataFrame(
    [known_row, masked_row, dict.fromkeys(MODEL_FEATURE_NAMES, "Unknown")],
    columns=list(MODEL_FEATURE_NAMES),
)

timing_rows = []
for model_name, estimator in [
    ("RandomForest_base", model_rf),
    ("RandomForest_tuned", model_rf_tuned),
]:
    timing_input = self_check_X.iloc[[1]]
    estimator.predict_proba(timing_input)  # 최초 준비 시간을 제외한 반복 추론을 측정한다.
    durations = []
    for _ in range(30):
        started = perf_counter()
        estimator.predict_proba(timing_input)
        durations.append((perf_counter() - started) * 1000)
    timing_rows.append(
        {
            "model": model_name,
            "warm_predict_mean_ms": np.mean(durations),
            "warm_predict_p95_ms": np.percentile(durations, 95),
        }
    )
timing_comparison = pd.DataFrame(timing_rows).set_index("model")
display(timing_comparison.round(3))

# 비교가 끝난 Train 학습 모델만 별도 후보 파일로 저장한다. 기존 Stacking 파일은 건드리지 않는다.
artifact_path = (
    preprocessing_notebook.parents[2]
    / "backend"
    / "pipeline"
    / "artifacts"
    / "deal-random-forest-v1.joblib"
)
artifact_path.parent.mkdir(parents=True, exist_ok=True)
bundle = {
    "schema_version": 1,
    "model_version": "deal-random-forest-v1",
    "model": model_rf_tuned,
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "category_values": {column: list(values) for column, values in CATEGORY_VALUES.items()},
    "target": {"Lost": 0, "Won": 1},
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "source_sha256": SOURCE_SHA256,
    "training_scope": "train_split_only",
    "training_original_rows": train_original_rows,
    "training_masked_rows": len(y_train),
    "best_params": search_rf.best_params_,
    "cv_brier": -search_rf.best_score_,
    "test_metrics_mean": test_tuned.mean().to_dict(),
    "versions": {name: version(name) for name in ("scikit-learn", "numpy", "pandas", "joblib")},
}
joblib.dump(bundle, artifact_path)
restored = joblib.load(artifact_path)
np.testing.assert_allclose(
    model_rf_tuned.predict_proba(self_check_X),
    restored["model"].predict_proba(self_check_X),
    rtol=1e-12,
    atol=1e-12,
)
assert np.isfinite(restored["model"].predict_proba(self_check_X)).all()
assert set(restored["model"].classes_) == {0, 1}
print(f"후보 저장: backend/pipeline/artifacts/{artifact_path.name}")
print(f"파일 크기: {artifact_path.stat().st_size / 1024**2:.3f} MiB")
print("저장 후 재로드 확률 검증: 통과")

,warm_predict_mean_ms,warm_predict_p95_ms
model,,
RandomForest_base,2.717,2.917
RandomForest_tuned,5.400,5.799


후보 저장: backend/pipeline/artifacts/deal-random-forest-v1.joblib
파일 크기: 7.535 MiB
저장 후 재로드 확률 검증: 통과


### 해석

추론 시간은 이 컴퓨터에서 모델을 이미 로드한 상태의 합성 입력 1건을 30회 예측한 값이다.
AWS 초기 로딩·네트워크·LLM 구조화 시간이나 동시 요청 대기시간을 포함하지 않는다.

`deal-random-forest-v1.joblib` 하나에 원핫 인코더·RandomForest와 입력 계약이 들어간다.
Test를 제외한 Train으로 학습된 **검토용 후보**이며, 기존 Stacking 서비스 로더와 파일 형식이 다르다.
AWS 모델이나 백엔드 추론 코드는 자동으로 교체하지 않는다. 학습 원본과 파일 산출물은 Git에 올리지 않는다.